In [12]:
# IMPORT LIBRARIES
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# Project Overview

This project builds a baseline machine learning model to detect fraudulent credit card transactions using the IEEE-CIS Fraud Detection dataset.

The goal is to create a simple but realistic end-to-end pipeline covering, data loading and merging, preprocessing, time based splitting, baseline model training (LightGBM) and evaluation using AUC.

### The Business Problem: How do we detect fraudulent or risky transactions before they cause financial loss?

In practical terms, financial platforms (banks, fintech apps, payment processors) process millions of transactions in real time. Some of these transactions are either fraudulent (stolen cards, account takeovers) or risky (likely to default and lead to loss, misuse of system e.g. exploiting promos, fake accounts).

Blocking fraud before it happens helps financial institutions reduce financial losses from fraud, flag suspicious transactions in real time and improve customer trust and security. 

In [13]:
# LOAD DATA
train_id = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
train_trans = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train = train_trans.merge(train_id, on = 'TransactionID', how = 'left')

## Dataset Description

The dataset comes from the IEEE-CIS Fraud Detection competition and contains two main tables:

**1. Transaction Data (train_transaction.csv)** which contains transaction-level information such as:

    * TransactionAmt
    * TransactionDT (time delta of transaction)
    * ProductCD (product category)
    * Card and behavioral features (card1, C1, D1, V1, etc.)
    * Target variable: isFraud (1 = fraud, 0 = legitimate) etc.

**2. Identity Data (train_identity.csv)** which contains additional device and identity information such as:

    * DeviceType
    * DeviceInfo
    * network and browser-related features etc.

## Data exploration

In [14]:
train.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


The dataset is clearly missing lots of numerical info as proven by all the NaNs. This will need to be cleaned using median. Median is used because unlike mean, it does not get skewed by outliers and filling with median keeps the filled value typical of the data. Filling with 0 is not used because the model can interpret it as an actual value which is not ideal. 

Additionallly, columns like ProductCD, card4, id_31 etc contain categorical variables. While LightGBM can handle this, it is still preferable to encode the categories as integers. 

In [15]:
# Check correlation with the target to inform feature selection decison 
correlation = train.select_dtypes(include=['number']).corr()['isFraud'].sort_values(ascending=False) 
pd.set_option('display.max_rows', None)
print(correlation)
pd.reset_option('display.max_rows')

isFraud           1.000000
V257              0.383060
V246              0.366878
V244              0.364129
V242              0.360590
V201              0.328005
V200              0.318783
V189              0.308219
V188              0.303582
V258              0.297151
V45               0.281832
V158              0.278066
V156              0.275952
V149              0.273282
V228              0.268861
V44               0.260376
V86               0.251828
V87               0.251737
V170              0.249794
V147              0.242894
V52               0.239469
V157              0.234866
V155              0.234199
V230              0.231740
V199              0.231024
V148              0.228891
V51               0.223191
V171              0.216508
V40               0.212442
V243              0.210238
V154              0.206958
V190              0.205148
V39               0.203097
V38               0.199005
V146              0.198584
V43               0.198274
V140              0.196938
V

## Time aware split

In [16]:
# used domain knowledge + correlation (numeric only) to select features
baseline_features = ['TransactionAmt', 'card1', 'card3', 'card5', 'card6', 'C1', 'C2', 'C8', 'C10', 
            'C11', 'C12', 'D1', 'D2', 'D4', 'D10', 'D15', 'M1', 'M2', 'M3', 'M6', 'M7', 'M8', 'M9',  
            'V257', 'V246', 'V244', 'V242', 'V201', 'V45', 'V86', 'V87'] 

# TIME AWARE SPLIT
# first, sort by TransactionDT (time) to avoid future leakage
train_sorted = train.sort_values('TransactionDT').reset_index(drop=True)
X = train_sorted[baseline_features].copy()
y = train_sorted['isFraud'].copy()

# next, split; test is the most recent 20%
test_size = int(len(X) * 0.2)
X_temp = X.iloc[:-test_size].copy()
X_test = X.iloc[-test_size:].copy() 
y_temp = y.iloc[:-test_size].copy()
y_test = y.iloc[-test_size:].copy()

# val is the most recent 20% of remaining 80%
val_size = int(len(X_temp) * 0.2)
X_train = X_temp.iloc[:-val_size].copy()
X_val = X_temp.iloc[-val_size:].copy()
y_train = y_temp.iloc[:-val_size].copy()
y_val = y_temp.iloc[-val_size:].copy()
'''written in separate lines like because if something breaks, it's easier to inspect each step (debugging)'''

# compute fraud rate for each set
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Train fraud rate:  {y_train.mean():.4f}")
print(f"Val fraud rate:    {y_val.mean():.4f}")
print(f'Test fraud rate:   {y_test.mean():.4f}')
print(baseline_features)

Train: 377,946 | Val: 94,486 | Test: 118,108
Train fraud rate:  0.0341
Val fraud rate:    0.0392
Test fraud rate:   0.0344
['TransactionAmt', 'card1', 'card3', 'card5', 'card6', 'C1', 'C2', 'C8', 'C10', 'C11', 'C12', 'D1', 'D2', 'D4', 'D10', 'D15', 'M1', 'M2', 'M3', 'M6', 'M7', 'M8', 'M9', 'V257', 'V246', 'V244', 'V242', 'V201', 'V45', 'V86', 'V87']


### Class Imbalance

* The dataset is clearly imbalanced with ~3.5% fraud in all three sets.
* This means that scale_pos_weight in LightGM will be 364721 / 13224 ≈ 27.6. Meaning that fraud matters 27x more than not fraud.
* This will cause an issue when training because the model will learn to minimize loss by predicting low probabailities (not fraud) most of the time and still have low error/accuracy because it predicts correctly for the majority class.
* Therefore, the resulting AUC will be over optimistic and the model will fail to generalize on new data.
* To fix this, class_weight must be a model parameter. 

## Preprocessing

In [17]:
#  Use median imputation to replace NaNs in the numerical columns that are in the training data  
imp = SimpleImputer(strategy='median') 
num_columns = X_train.select_dtypes(include = 'number').columns.tolist() 
X_train[num_columns] = imp.fit_transform(X_train[num_columns])
X_val[num_columns] = imp.transform(X_val[num_columns])
X_test[num_columns]  = imp.transform(X_test[num_columns])

# Helper function for encoding categorical variables in train, val and test as numeric because LightGBM only accepts numeric data
def encode_objects(df_train, df_val): 
    df_train = df_train.copy()
    df_val   = df_val.copy()

    for col in df_train.select_dtypes(include='object').columns: 
        df_train[col] = df_train[col].astype('category') 
        df_val[col] = df_val[col].astype('category') 
    return df_train , df_val 

X_train, X_val = encode_objects(X_train, X_val)

categorical_cols = X_train.select_dtypes(include='category').columns.tolist()

## Model training

In [18]:
# BASELINE MODEL - LIGHTGBM
model = lgb.LGBMClassifier(n_estimators=300, max_depth=5, class_weight='balanced', random_state=42)
model.fit(X_train, y_train, categorical_feature = categorical_cols) # 

# 7. PREDICT AND EVALUATE 
train_preds = model.predict_proba(X_train)[:, 1]
val_preds   = model.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, train_preds)
val_auc = roc_auc_score(y_val, val_preds)

print("Train AUC:", round(train_auc, 4))
print("Val/Baseline AUC:", round(val_auc, 4))
print("Gap:", round(train_auc - val_auc, 4))

[LightGBM] [Info] Number of positive: 12895, number of negative: 365051
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.047259 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3671
[LightGBM] [Info] Number of data points in the train set: 377946, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

* Train AUC is high (0.9314) meaning that the model fits training data very well
* Val AUC is lower (0.8702) meaning that the performance drops on unseen data. The validation AUC serves as the baseline metric against which future model improvements will be evaluated.
* Gap (~0.06) means that the model is moderately overfitting. This can be improved with better validation or tuning.

In [19]:
# Log experiment 
import datetime

log = pd.DataFrame([{
    'date': datetime.date.today(),
    'model': 'LightGBM_baseline',
    'n_features': len(baseline_features),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'roc_auc': round(val_auc, 4),
}])

log.to_csv('experiment_log.csv', index=False)
print(log)

         date              model  n_features  train_size  val_size  roc_auc
0  2026-04-16  LightGBM_baseline          31      377946     94486   0.8702
